# Stage 04x — Model Comparison

**Purpose:** Compare three parallel model-building approaches (MIV, XGBoost Importance, Forward Stepwise), compute weighted composite scores, and select the champion model.

**Inputs:**
- `{RUN_DIR}/pipeline/model_params_miv.json`
- `{RUN_DIR}/pipeline/model_params_xgb.json`
- `{RUN_DIR}/pipeline/model_params_fwd.json`
- `{RUN_DIR}/data/loans_binned.csv`

In [ ]:
import sys, os
PROJECT_ROOT = r'C:/projects/superagent'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
import pdtoolkit as pdt
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score
import shutil

RUN_DIR = 'runs/2026-03-17_071354'

# Colour palette
BLUE = '#2166AC'
RED = '#D6604D'
GREY = '#999999'

# Model colours for comparison
MIV_COLOR = '#1f77b4'
XGB_COLOR = '#ff7f0e'
FWD_COLOR = '#2ca02c'

In [ ]:
# Load model parameters
with open(f'{RUN_DIR}/pipeline/model_params_miv.json', 'r') as f:
    params_miv = json.load(f)
with open(f'{RUN_DIR}/pipeline/model_params_xgb.json', 'r') as f:
    params_xgb = json.load(f)
with open(f'{RUN_DIR}/pipeline/model_params_fwd.json', 'r') as f:
    params_fwd = json.load(f)

# Load binned dataset
df_binned = pd.read_csv(f'{RUN_DIR}/data/loans_binned.csv')
y = df_binned['Creditability'].values

print(f'Binned dataset: {df_binned.shape[0]} rows, {df_binned.shape[1]} columns')
print(f'Default rate: {y.mean():.4f}')

In [ ]:
# Extract metrics for each model
models = {
    'MIV': {
        'params': params_miv,
        'auc': params_miv['model_auc'],
        'gini': params_miv['model_gini'],
        'ks': params_miv['model_ks'],
        'n_vars': len(params_miv['selected_variables']),
        'variables': params_miv['selected_variables'],
        'pseudo_r2': 0.2222,  # from stage summary
        'cv_auc_gap': 0.0044,  # from stage summary
        'signs_consistent': True,  # all true from stage summary
        'decile_monotonic': True,  # from stage summary
    },
    'XGBoost': {
        'params': params_xgb,
        'auc': params_xgb['model_auc'],
        'gini': params_xgb['model_gini'],
        'ks': params_xgb['model_ks'],
        'n_vars': len(params_xgb['selected_variables']),
        'variables': params_xgb['selected_variables'],
        'pseudo_r2': 0.2184,  # from stage summary
        'cv_auc_gap': 0.0049,  # from stage summary
        'signs_consistent': True,  # all true from stage summary
        'decile_monotonic': True,  # from stage summary
    },
    'Forward': {
        'params': params_fwd,
        'auc': params_fwd['model_auc'],
        'gini': params_fwd['model_gini'],
        'ks': params_fwd['model_ks'],
        'n_vars': len(params_fwd['selected_variables']),
        'variables': params_fwd['selected_variables'],
        'pseudo_r2': 0.2222,  # from stage summary
        'cv_auc_gap': 0.0044,  # from stage summary
        'signs_consistent': True,  # all true from stage summary
        'decile_monotonic': True,  # from stage summary
    }
}

# Display raw metrics comparison
metrics_table = pd.DataFrame({
    'Metric': ['Selected Variables', 'AUC', 'Gini', 'KS', 'Pseudo R-squared', 'CV AUC Gap', 'Signs Consistent', 'Decile Monotonic'],
    'MIV': [8, 0.8109, 0.6218, 0.5067, 0.2222, 0.0044, 'Yes', 'Yes'],
    'XGBoost': [7, 0.807, 0.614, 0.4886, 0.2184, 0.0049, 'Yes', 'Yes'],
    'Forward': [8, 0.8109, 0.6218, 0.5067, 0.2222, 0.0044, 'Yes', 'Yes']
}).set_index('Metric')

print('=== Raw Metrics Comparison ===')
print(metrics_table.to_string())

In [ ]:
# Compute weighted composite scores
# Step 1: Min-max normalize AUC, Gini, KS across models

def minmax_normalize(values):
    """Min-max normalize. If all equal, all get 1.0."""
    v = np.array(values, dtype=float)
    vmin, vmax = v.min(), v.max()
    if vmax - vmin < 1e-10:
        return np.ones_like(v)
    return (v - vmin) / (vmax - vmin)

auc_vals = [models[m]['auc'] for m in ['MIV', 'XGBoost', 'Forward']]
gini_vals = [models[m]['gini'] for m in ['MIV', 'XGBoost', 'Forward']]
ks_vals = [models[m]['ks'] for m in ['MIV', 'XGBoost', 'Forward']]

auc_norm = minmax_normalize(auc_vals)
gini_norm = minmax_normalize(gini_vals)
ks_norm = minmax_normalize(ks_vals)

print('Normalized AUC:', dict(zip(['MIV', 'XGBoost', 'Forward'], auc_norm)))
print('Normalized Gini:', dict(zip(['MIV', 'XGBoost', 'Forward'], gini_norm)))
print('Normalized KS:', dict(zip(['MIV', 'XGBoost', 'Forward'], ks_norm)))

In [ ]:
# Step 2: Compute sub-scores for each criterion

composite_details = {}

for i, name in enumerate(['MIV', 'XGBoost', 'Forward']):
    m = models[name]
    
    # CV stability: 1.0 - (|dev_auc - cv_auc| / 0.03), capped at [0, 1]
    cv_stability = max(0.0, min(1.0, 1.0 - (m['cv_auc_gap'] / 0.03)))
    
    # Coefficient sign consistency: 1.0 if all consistent
    sign_consistency = 1.0 if m['signs_consistent'] else 0.0
    
    # Parsimony: 1.0 - (n_vars - 4) / (12 - 4), capped at [0, 1]
    parsimony = max(0.0, min(1.0, 1.0 - (m['n_vars'] - 4) / (12 - 4)))
    
    # Decile monotonicity
    monotonicity = 1.0 if m['decile_monotonic'] else 0.0
    
    # Weighted composite
    weights = {
        'auc': 0.25,
        'gini': 0.15,
        'ks': 0.10,
        'cv_stability': 0.20,
        'sign_consistency': 0.15,
        'parsimony': 0.10,
        'monotonicity': 0.05
    }
    
    sub_scores = {
        'auc': auc_norm[i],
        'gini': gini_norm[i],
        'ks': ks_norm[i],
        'cv_stability': cv_stability,
        'sign_consistency': sign_consistency,
        'parsimony': parsimony,
        'monotonicity': monotonicity
    }
    
    composite = sum(weights[k] * sub_scores[k] for k in weights)
    
    composite_details[name] = {
        'sub_scores': sub_scores,
        'weighted': {k: weights[k] * sub_scores[k] for k in weights},
        'composite': composite
    }
    
    print(f'\n=== {name} ===')
    for k in weights:
        print(f'  {k:20s}: raw={sub_scores[k]:.4f}  weighted={weights[k] * sub_scores[k]:.4f}')
    print(f'  {"COMPOSITE":20s}: {composite:.4f}')

In [ ]:
# Reconstruct predicted probabilities for ROC overlay

def reconstruct_predictions(params, df):
    """Reconstruct logistic regression predictions from binned data + WoE mappings."""
    woe_encoded = pd.DataFrame(index=df.index)
    
    for var in params['selected_variables']:
        # Build bin-label -> WoE lookup from model params
        woe_map = {entry['bin']: entry['woe'] for entry in params['woe_mappings'][var]}
        woe_encoded[var] = df[var].map(woe_map)
    
    # Check for unmapped bins
    n_missing = woe_encoded.isna().sum().sum()
    if n_missing > 0:
        print(f'  WARNING: {n_missing} unmapped bin values')
        woe_encoded = woe_encoded.fillna(0.0)
    
    X = woe_encoded.values
    coefs = np.array([params['coefficients'][v] for v in params['selected_variables']])
    intercept = params['intercept']
    
    logit = intercept + X @ coefs
    prob = 1.0 / (1.0 + np.exp(-logit))
    return prob

pred_miv = reconstruct_predictions(params_miv, df_binned)
pred_xgb = reconstruct_predictions(params_xgb, df_binned)
pred_fwd = reconstruct_predictions(params_fwd, df_binned)

# Verify AUCs match
for name, pred in [('MIV', pred_miv), ('XGBoost', pred_xgb), ('Forward', pred_fwd)]:
    auc = roc_auc_score(y, pred)
    print(f'{name} reconstructed AUC: {auc:.4f}')

In [ ]:
# Plot 1: ROC Curve Overlay

fig, ax = plt.subplots(figsize=(8, 8))

for label, pred, color, auc_val in [
    ('MIV', pred_miv, MIV_COLOR, models['MIV']['auc']),
    ('XGBoost', pred_xgb, XGB_COLOR, models['XGBoost']['auc']),
    ('Forward', pred_fwd, FWD_COLOR, models['Forward']['auc'])
]:
    fpr, tpr, _ = roc_curve(y, pred)
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc_val:.4f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison \u2014 Three Selection Methods')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/04x_roc_overlay.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04x_roc_overlay.png')

In [ ]:
# Plot 2: Metrics Comparison Bar Chart + Composite Score Breakdown

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Discrimination metrics
disc_metrics = pd.DataFrame({
    'MIV': [models['MIV']['auc'], models['MIV']['gini'], models['MIV']['ks']],
    'XGBoost': [models['XGBoost']['auc'], models['XGBoost']['gini'], models['XGBoost']['ks']],
    'Forward': [models['Forward']['auc'], models['Forward']['gini'], models['Forward']['ks']]
}, index=['AUC', 'Gini', 'KS'])

disc_metrics.plot(kind='bar', ax=axes[0], color=[MIV_COLOR, XGB_COLOR, FWD_COLOR], edgecolor='white')
axes[0].set_title('Discrimination Metrics')
axes[0].set_ylabel('Value')
axes[0].set_ylim(0.4, 0.9)
axes[0].legend(loc='upper left')
axes[0].tick_params(axis='x', rotation=0)

# Right: Composite score stacked bar (weighted contributions)
criteria = ['auc', 'gini', 'ks', 'cv_stability', 'sign_consistency', 'parsimony', 'monotonicity']
criteria_labels = ['AUC (25%)', 'Gini (15%)', 'KS (10%)', 'CV Stab. (20%)', 'Sign Con. (15%)', 'Parsimony (10%)', 'Monoton. (5%)']
model_names = ['MIV', 'XGBoost', 'Forward']

bottom = np.zeros(3)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']

for j, (crit, label) in enumerate(zip(criteria, criteria_labels)):
    vals = [composite_details[m]['weighted'][crit] for m in model_names]
    axes[1].bar(model_names, vals, bottom=bottom, label=label, color=colors[j], edgecolor='white')
    bottom += np.array(vals)

# Add composite score text on top
for i, m in enumerate(model_names):
    axes[1].text(i, composite_details[m]['composite'] + 0.01, 
                 f"{composite_details[m]['composite']:.4f}", ha='center', fontweight='bold')

axes[1].set_title('Composite Score Breakdown')
axes[1].set_ylabel('Weighted Score')
axes[1].legend(loc='upper left', fontsize=8)
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/04x_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04x_model_comparison.png')

In [ ]:
# Plot 3: Variable Overlap Analysis

miv_vars = set(params_miv['selected_variables'])
xgb_vars = set(params_xgb['selected_variables'])
fwd_vars = set(params_fwd['selected_variables'])

all_vars = sorted(miv_vars | xgb_vars | fwd_vars)
overlap_df = pd.DataFrame({
    'MIV': [1 if v in miv_vars else 0 for v in all_vars],
    'XGBoost': [1 if v in xgb_vars else 0 for v in all_vars],
    'Forward': [1 if v in fwd_vars else 0 for v in all_vars]
}, index=all_vars)

fig, ax = plt.subplots(figsize=(10, max(6, len(all_vars) * 0.5)))
sns.heatmap(overlap_df, annot=True, cmap='YlGn', cbar=False, ax=ax,
            linewidths=0.5, linecolor='white', fmt='d')
ax.set_title('Variable Selection Overlap')
ax.set_ylabel('Variable')
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/04x_variable_overlap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: 04x_variable_overlap.png')

# Compute overlap statistics
intersection = miv_vars & xgb_vars & fwd_vars
union = miv_vars | xgb_vars | fwd_vars
overlap_ratio = len(intersection) / len(union)

print(f'\nVariables selected by all three: {sorted(intersection)}')
print(f'Variables unique to MIV: {sorted(miv_vars - xgb_vars - fwd_vars)}')
print(f'Variables unique to XGBoost: {sorted(xgb_vars - miv_vars - fwd_vars)}')
print(f'Variables unique to Forward: {sorted(fwd_vars - miv_vars - xgb_vars)}')
print(f'Overlap ratio (|intersection|/|union|): {overlap_ratio:.4f}')

In [ ]:
# Champion Selection

scores = {m: composite_details[m]['composite'] for m in model_names}
ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

champion_name = ranked[0][0]
champion_score = ranked[0][1]
runner_up_name = ranked[1][0]
runner_up_score = ranked[1][1]

print(f'\n=== CHAMPION SELECTION ===')
for rank, (name, score) in enumerate(ranked, 1):
    marker = ' <<< CHAMPION' if rank == 1 else ''
    print(f'  #{rank}: {name} — composite score: {score:.4f}{marker}')

# Map display name to suffix
suffix_map = {'MIV': 'miv', 'XGBoost': 'xgb', 'Forward': 'fwd'}
champion_suffix = suffix_map[champion_name]
runner_up_suffix = suffix_map[runner_up_name]

# Note: MIV and Forward produce identical models (same variables, same coefficients)
# MIV is selected as champion by convention (alphabetically first among ties)
# since the composite scores are identical.
print(f'\nNote: MIV and Forward selected identical variables and produced identical')
print(f'coefficients. MIV is chosen as champion (first among equal-scoring methods).')

In [ ]:
# Copy champion model_params to canonical path

champion_file = f'{RUN_DIR}/pipeline/model_params_{champion_suffix}.json'
canonical_file = f'{RUN_DIR}/pipeline/model_params.json'
shutil.copy2(champion_file, canonical_file)
print(f'Copied {champion_file} -> {canonical_file}')

## Stage Summary

| Item | Value | Status |
|---|---|---|
| Models compared | 3 (MIV, XGBoost, Forward) | PASS |
| Champion | MIV (composite: 0.9207) | PASS |
| Champion AUC | 0.8109 | PASS |
| Champion Gini | 0.6218 | PASS |
| Champion KS | 0.5067 | PASS |
| Variable overlap ratio | 0.875 (7 of 8 vars shared by all) | PASS |
| MIV = Forward identical | Yes (same vars, same coefficients) | WARN |

**Flags for human review:** MIV and Forward Stepwise produced identical models (same 8 variables, identical coefficients). This is expected when MIV threshold and forward p-value threshold both admit the same variables from the same shortlist. XGBoost Importance excluded Credit Amount, resulting in marginally lower AUC (0.807 vs 0.8109).

**Recommended action for next stage:** Proceed to Stage 05 (Calibration) using the champion model parameters at `{RUN_DIR}/pipeline/model_params.json`.